<a href="https://colab.research.google.com/github/Navya40869/edge-idps-colab/blob/main/risk_engine/05_risk_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

# ==========================================
# 1. FUZZY MEMBERSHIP FUNCTIONS
# ==========================================
def fuzzy_confidence(conf):
    """
    Maps continuous confidence percentage (0 - 100%) to fuzzy membership degrees.
    Returns fuzzy sets: Low, Medium, High.
    """
    # Low confidence membership (triangular/trapezoidal)
    low = max(0.0, min(1.0, (60.0 - conf) / 30.0)) if conf <= 60 else 0.0

    # Medium confidence membership
    medium = max(0.0, min((conf - 40.0) / 20.0, (80.0 - conf) / 20.0)) if 40 <= conf <= 80 else 0.0

    # High confidence membership
    high = max(0.0, min(1.0, (conf - 70.0) / 30.0)) if conf >= 70 else 0.0

    return {"Low": low, "Medium": medium, "High": high}


def get_attack_base_severity(predicted_class):
    """
    Categorizes the attack class into a baseline severity multiplier (0 to 3).
    """
    cls = str(predicted_class).lower()

    if "benign" in cls or "normal" in cls or cls == "0":
        return 0  # Safe Traffic
    elif any(k in cls for k in ["recon", "scan", "ping"]):
        return 1  # Low Threat (Reconnaissance)
    elif any(k in cls for k in ["spoofing", "bruteforce", "ack"]):
        return 2  # Medium Threat (Access / Manipulation)
    else:
        return 3  # High Threat (DDoS, Floods, Malware)

# ==========================================
# 2. RULE ENGINE & RISK SCORE GENERATOR
# ==========================================
def evaluate_risk(predicted_class, confidence_score):
    """
    Evaluates Fuzzy Rules and outputs Risk Score (0-10), Level, and Action.
    """
    # 1. Calculate Base Severity and Fuzzy Confidence
    base_severity = get_attack_base_severity(predicted_class)
    conf_fuzzy = fuzzy_confidence(confidence_score)

    # Normal traffic gets 0 risk instantly
    if base_severity == 0:
        return 0.0, "LOW", "ALLOW"

    # 2. Fuzzy Defuzzification / Weighted Confidence Multiplier
    conf_weight = (conf_fuzzy["Low"] * 0.4) + (conf_fuzzy["Medium"] * 0.75) + (conf_fuzzy["High"] * 1.0)

    # 3. Calculate Risk Score (Scale: 0.0 to 10.0)
    raw_risk = (base_severity * 2.9) * conf_weight
    risk_score = round(float(np.clip(raw_risk, 1.0, 10.0)), 2)

    # 4. Rule Engine for Decision Action Mapping
    if risk_score < 3.0:
        risk_level = "LOW"
        action = "LOG_WARNING"
    elif risk_score < 6.0:
        risk_level = "MEDIUM"
        action = "THROTTLE_BANDWIDTH"
    elif risk_score < 8.5:
        risk_level = "HIGH"
        action = "DROP_PACKET"
    else:
        risk_level = "CRITICAL"
        action = "BLOCK_IP"

    return risk_score, risk_level, action

# ==========================================
# 3. VALIDATION & TEST SUITE FOR MEMBER 5
# ==========================================
test_cases = [
    ("BENIGN", 99.1),
    ("Recon-PortScan", 45.0),    # Low severity + low confidence
    ("Recon-PortScan", 95.0),    # Low severity + high confidence
    ("DDoS-UDP_Flood", 52.0),    # High severity + low confidence
    ("DDoS-UDP_Flood", 98.6)     # High severity + high confidence
]

print("="*60)
print("   MEMBER 5: FUZZY RISK ENGINE VALIDATION RESULTS   ")
print("="*60)

for atk, conf in test_cases:
    score, level, action = evaluate_risk(atk, conf)
    print(f"Class: {atk:<16} | Conf: {conf:>5.1f}% | Risk: {score:>4.1f}/10 | Level: {level:<8} | Action: {action}")

   MEMBER 5: FUZZY RISK ENGINE VALIDATION RESULTS   
Class: BENIGN           | Conf:  99.1% | Risk:  0.0/10 | Level: LOW      | Action: ALLOW
Class: Recon-PortScan   | Conf:  45.0% | Risk:  1.1/10 | Level: LOW      | Action: LOG_WARNING
Class: Recon-PortScan   | Conf:  95.0% | Risk:  2.4/10 | Level: LOW      | Action: LOG_WARNING
Class: DDoS-UDP_Flood   | Conf:  52.0% | Risk:  4.8/10 | Level: MEDIUM   | Action: THROTTLE_BANDWIDTH
Class: DDoS-UDP_Flood   | Conf:  98.6% | Risk:  8.3/10 | Level: HIGH     | Action: DROP_PACKET


In [ ]:
%%writefile risk_engine.py
import numpy as np

def fuzzy_confidence(conf):
    low = max(0.0, min(1.0, (60.0 - conf) / 30.0)) if conf <= 60 else 0.0
    medium = max(0.0, min((conf - 40.0) / 20.0, (80.0 - conf) / 20.0)) if 40 <= conf <= 80 else 0.0
    high = max(0.0, min(1.0, (conf - 70.0) / 30.0)) if conf >= 70 else 0.0
    return {"Low": low, "Medium": medium, "High": high}

def get_attack_base_severity(predicted_class):
    cls = str(predicted_class).lower()
    if "benign" in cls or "normal" in cls or cls == "0":
        return 0
    elif any(k in cls for k in ["recon", "scan", "ping"]):
        return 1
    elif any(k in cls for k in ["spoofing", "bruteforce", "ack"]):
        return 2
    else:
        return 3

def evaluate_risk(predicted_class, confidence_score):
    base_severity = get_attack_base_severity(predicted_class)
    conf_fuzzy = fuzzy_confidence(confidence_score)

    if base_severity == 0:
        return 0.0, "LOW", "ALLOW"

    conf_weight = (conf_fuzzy["Low"] * 0.4) + (conf_fuzzy["Medium"] * 0.75) + (conf_fuzzy["High"] * 1.0)
    raw_risk = (base_severity * 2.9) * conf_weight
    risk_score = round(float(np.clip(raw_risk, 1.0, 10.0)), 2)

    if risk_score < 3.0:
        risk_level, action = "LOW", "LOG_WARNING"
    elif risk_score < 6.0:
        risk_level, action = "MEDIUM", "THROTTLE_BANDWIDTH"
    elif risk_score < 8.5:
        risk_level, action = "HIGH", "DROP_PACKET"
    else:
        risk_level, action = "CRITICAL", "BLOCK_IP"

    return risk_score, risk_level, action

Writing risk_engine.py


In [1]:
import os
from google.colab import drive

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Define path to write the .py file
target_path = "/content/drive/MyDrive/Edge-IDPS-Data/processed_data/risk_engine.py"

# 3. Code content for the executable python script
risk_engine_code = """import numpy as np

def fuzzy_confidence(conf):
    low = max(0.0, min(1.0, (60.0 - conf) / 30.0)) if conf <= 60 else 0.0
    medium = max(0.0, min((conf - 40.0) / 20.0, (80.0 - conf) / 20.0)) if 40 <= conf <= 80 else 0.0
    high = max(0.0, min(1.0, (conf - 70.0) / 30.0)) if conf >= 70 else 0.0
    return {"Low": low, "Medium": medium, "High": high}

def get_attack_base_severity(predicted_class):
    cls = str(predicted_class).lower()
    if "benign" in cls or "normal" in cls or cls == "0":
        return 0
    elif any(k in cls for k in ["recon", "scan", "ping"]):
        return 1
    elif any(k in cls for k in ["spoofing", "bruteforce", "ack"]):
        return 2
    else:
        return 3

def evaluate_risk(predicted_class, confidence_score):
    base_severity = get_attack_base_severity(predicted_class)
    conf_fuzzy = fuzzy_confidence(confidence_score)

    if base_severity == 0:
        return 0.0, "LOW", "ALLOW"

    conf_weight = (conf_fuzzy["Low"] * 0.4) + (conf_fuzzy["Medium"] * 0.75) + (conf_fuzzy["High"] * 1.0)
    raw_risk = (base_severity * 2.9) * conf_weight
    risk_score = round(float(np.clip(raw_risk, 1.0, 10.0)), 2)

    if risk_score < 3.0:
        risk_level, action = "LOW", "LOG_WARNING"
    elif risk_score < 6.0:
        risk_level, action = "MEDIUM", "THROTTLE_BANDWIDTH"
    elif risk_score < 8.5:
        risk_level, action = "HIGH", "DROP_PACKET"
    else:
        risk_level, action = "CRITICAL", "BLOCK_IP"

    return risk_score, risk_level, action
"""

# 4. Write directly to Drive
with open(target_path, "w") as f:
    f.write(risk_engine_code)

print(f"✅ Successfully created {target_path}!")

Mounted at /content/drive
✅ Successfully created /content/drive/MyDrive/Edge-IDPS-Data/processed_data/risk_engine.py!
